# Zonal_mpsi using UXarray

Work on [NCL's zonal_mpsi](https://www.ncl.ucar.edu/Document/Functions/Built-in/zonal_mpsi.shtml) with a UXarray dataset.

Zonal mean meridional stream function

*v*: A multi-dimensional array of meridional wind values

*lat*: A one-dimensional array of latitudes.

*p*: A one-dimensional array of pressure level values ordered top-to-bottom.

*ps*: A multi-dimensional numeric array of surface pressures.

$MPSI(lev,lat) = \frac{2 {\pi} a cos(lat)}{g} \int_{p}^{PS} \bar{v} \,dp$

In [3]:
import uxarray as ux
import xarray as xr
import numpy as np
import geocat.datafiles as gdf
import geocat.comp as gc

/Users/jkent/miniconda3/envs/uxarray-where/lib/python3.13/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


Need a datset with: wind speed values (lat, long, time, maybe lev), vertical pressure levels, and spatial surface pressure

## Read in Data

In [5]:
data_path = 'netcdf_files/e2p3b09.F2000climo.ne30pg3.ctl002.cam.h0.0005-01.zonal_mpsi_subset.nc'
grid_path = 'netcdf_files/ne30pg3_scrip_170604.nc'
uxds = ux.open_dataset(gdf.get(grid_path), gdf.get(data_path)).drop('lat') #dropping lat to avoid using it instead of the UXarray grid

uxds

/var/folders/dd/_xm_pbpd3flgbvbnt7qhd70snnbpj_/T/ipykernel_57867/1643203269.py:3: DeprecationWarning: dropping variables using `drop` is deprecated; use drop_vars.
  uxds = ux.open_dataset(gdf.get(grid_path), gdf.get(data_path)).drop('lat') #dropping lat to avoid using it instead of the UXarray grid


<xarray.UxDataset> Size: 6MB
Dimensions:  (time: 1, n_face: 48600, lev: 32, ilev: 33)
Coordinates:
  * time     (time) object 8B 0005-02-01 00:00:00
  * lev      (lev) float64 256B 3.643 7.595 14.36 24.61 ... 957.5 976.3 992.6
  * ilev     (ilev) float64 264B 2.255 5.032 10.16 18.56 ... 967.5 985.1 1e+03
Dimensions without coordinates: n_face
Data variables:
    PS       (time, n_face) float32 194kB ...
    V        (time, lev, n_face) float32 6MB ...
    hyam     (lev) float64 256B ...
    hybm     (lev) float64 256B ...
    hyai     (ilev) float64 264B ...
    hybi     (ilev) float64 264B ...

The data hasn't been interpolated to pressure levels (still on hybrid sigma levels)

In [6]:
# Grabbing correct data variables, confirming long names
print(#uxds.lat.attrs['long_name'],', ',
      uxds.PS.attrs['long_name'],', ',
      uxds.V.attrs['long_name'],', ',
      uxds.hyam.attrs['long_name'],', ', 
      uxds.hybm.attrs['long_name'],', ',
      uxds.hyai.attrs['long_name'],', ', 
      uxds.hybi.attrs['long_name'])

Surface pressure ,  Meridional wind ,  hybrid A coefficient at layer midpoints ,  hybrid B coefficient at layer midpoints ,  hybrid A coefficient at layer interfaces ,  hybrid B coefficient at layer interfaces


## Converting from sigma hybrid coordinates to Pressure levels

$p(k) = hya(k) * p_0 + hyb(k) * p_{surface}$

Source: [NCL `sigma2hybrid` documentation](https://www.ncl.ucar.edu/Document/Functions/Built-in/sigma2hybrid.shtml)


['geocat_comp.delta_pressure()`](https://geocat-comp.readthedocs.io/en/latest/user_api/generated/geocat.comp.meteorology.delta_pressure.html)

['geocat_comp.dpres_plev()`](https://geocat-comp.readthedocs.io/en/latest/user_api/generated/geocat.comp.meteorology.dpres_plev.html)

['geocat_comp.interp_hybrid_to_pressure()'](https://geocat-comp.readthedocs.io/en/latest/user_api/generated/geocat.comp.interpolation.interp_hybrid_to_pressure.html)

In [7]:
da_ipress = gc.interpolation.interp_hybrid_to_pressure(
    uxds.V, 
    uxds.PS, 
    uxds.hyam, 
    uxds.hybm)
da_ipress

TypeError: psfc, hya, and hyb must be xarray DataArrays or all numpy arrays

## interp_hybrid_to_pressure debugging

In [11]:
new_levels = np.array(
    [
        1000,
        925,
        850,
        700,
        500,
        400,
        300,
        250,
        200,
        150,
        100,
        70,
        50,
        30,
        20,
        10,
        7,
        5,
        3,
        2,
        1,
    ]
).astype(np.float32)  # Mandatory pressure levels (mb)

In [15]:
in_types = []
for i in [uxds.V, uxds.PS, uxds.hyam, uxds.hybm, new_levels]:
    it = type(i)
    in_types.append(it)

print(in_types)
print(len(set(in_types[:-1])))

[<class 'uxarray.core.dataarray.UxDataArray'>, <class 'uxarray.core.dataarray.UxDataArray'>, <class 'uxarray.core.dataarray.UxDataArray'>, <class 'uxarray.core.dataarray.UxDataArray'>, <class 'numpy.ndarray'>]
1


In [18]:
if not len(set(in_types[:-1])) == 1 or not isinstance(uxds.V, xr.DataArray):
    print('warning')

In [1]:
isinstance(uxds.V, xr.DataArray)

NameError: name 'uxds' is not defined

In [ ]:
type(uxds.V)

uxarray.core.dataarray.UxDataArray

In [8]:
in_types = [type(uxds.PS), type(uxds.hyam), type(uxds.hybm)]
if not set(in_types).issubset({xr.DataArray, np.ndarray}) or len(set(in_types)) > 1:
    print('warning')

warning


In [9]:
set(in_types).issubset({xr.DataArray, np.ndarray})

False

In [10]:
len(set(in_types)) > 1

False

In [12]:
if not set(in_types).issubset({xr.DataArray, np.ndarray, ux.core.dataarray.UxDataArray}) or len(set(in_types)) > 1:
    print